# Draco batch for Colab

This notebook mirrors the flow of `scripts/batch-draco.sh` but runs in Python so you can execute it directly from a Colab (or any Jupyter) session and keep using GPU resources for other work. Start by installing the Node dependency, then run the helper cells to quantize + Draco every GLB under `data/scene/scene`. Update the parameters and targets before executing the final cell.



In [1]:
import os
from pathlib import Path

if "REPO_ROOT" not in os.environ:
    default_root = Path("/content/drive/MyDrive/anki_ar")
    os.environ["REPO_ROOT"] = str(default_root)

print("REPO_ROOT:", os.environ["REPO_ROOT"])



REPO_ROOT: /content/drive/MyDrive/anki_ar


In [3]:
%%bash
set -euo pipefail
if [ -n "${REPO_ROOT:-}" ] && [ -f "$REPO_ROOT/package.json" ]; then
  ROOT="$REPO_ROOT"
else
  ROOT=$(python - <<'PY'
import os
path = os.getcwd()
while True:
    if os.path.exists(os.path.join(path, "package.json")):
        print(path)
        break
    parent = os.path.dirname(path)
    if parent == path:
        break
    path = parent
PY
)
fi
if [ -z "$ROOT" ] || [ ! -f "$ROOT/package.json" ]; then
  echo "Could not autodetect repo root. Falling back to parent directory..."
  ROOT=$(cd .. && pwd)
fi
if [ ! -f "$ROOT/package.json" ]; then
  echo "package.json vẫn không tìm thấy dưới $ROOT. Gán biến môi trường REPO_ROOT trước khi chạy cell này." >&2
  exit 1
fi
cd "$ROOT"
echo "Working directory: $PWD"
install_node() {
  apt-get update -y >/tmp/apt.log
  apt-get install -y curl gnupg >/tmp/apt.log
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash - >/tmp/node-setup.log
  apt-get install -y nodejs >/tmp/apt.log
}
if ! command -v npm >/dev/null 2>&1; then
  if install_node >/tmp/node-install.log 2>&1; then
    echo "Node installed via NodeSource"
  else
    cat /tmp/node-install.log
    echo "NodeSource install failed, falling back to apt nodejs/npm"
    apt-get install -y nodejs npm >/tmp/node-install-fallback.log 2>&1
    cat /tmp/node-install-fallback.log
  fi
fi
run_npm_install() {
  local attempt=0
  while true; do
    set +e
    npm install
    status=$?
    set -e
    if [ "$status" -eq 0 ]; then
      break
    fi
    attempt=$((attempt + 1))
    echo "npm install failed (attempt $attempt). Cleaning cache and retrying..."
    npm cache clean --force >/tmp/npm-cache.log 2>&1
    rm -rf node_modules package-lock.json
    if [ "$attempt" -ge 3 ]; then
      echo "npm install still failing after $attempt attempts. See /tmp/npm-cache.log and /root/.npm/_logs for details." >&2
      exit "$status"
    fi
  done
}
run_npm_install



Could not autodetect repo root. Falling back to parent directory...


package.json vẫn không tìm thấy dưới /. Gán biến môi trường REPO_ROOT trước khi chạy cell này.


CalledProcessError: Command 'b'set -euo pipefail\nif [ -n "${REPO_ROOT:-}" ] && [ -f "$REPO_ROOT/package.json" ]; then\n  ROOT="$REPO_ROOT"\nelse\n  ROOT=$(python - <<\'PY\'\nimport os\npath = os.getcwd()\nwhile True:\n    if os.path.exists(os.path.join(path, "package.json")):\n        print(path)\n        break\n    parent = os.path.dirname(path)\n    if parent == path:\n        break\n    path = parent\nPY\n)\nfi\nif [ -z "$ROOT" ] || [ ! -f "$ROOT/package.json" ]; then\n  echo "Could not autodetect repo root. Falling back to parent directory..."\n  ROOT=$(cd .. && pwd)\nfi\nif [ ! -f "$ROOT/package.json" ]; then\n  echo "package.json v\xe1\xba\xabn kh\xc3\xb4ng t\xc3\xacm th\xe1\xba\xa5y d\xc6\xb0\xe1\xbb\x9bi $ROOT. G\xc3\xa1n bi\xe1\xba\xbfn m\xc3\xb4i tr\xc6\xb0\xe1\xbb\x9dng REPO_ROOT tr\xc6\xb0\xe1\xbb\x9bc khi ch\xe1\xba\xa1y cell n\xc3\xa0y." >&2\n  exit 1\nfi\ncd "$ROOT"\necho "Working directory: $PWD"\ninstall_node() {\n  apt-get update -y >/tmp/apt.log\n  apt-get install -

In [ ]:
import csv
import subprocess
from datetime import datetime
from pathlib import Path
from typing import Optional


def find_repo_root() -> Path:
    path = Path.cwd()
    while True:
        if (path / "package.json").exists():
            return path.resolve()
        if path == path.parent:
            raise FileNotFoundError("Could not locate package.json; run this notebook from inside the repo.")
        path = path.parent


ROOT = find_repo_root()
BIN = ROOT / "node_modules" / ".bin" / "gltf-transform"
if not BIN.exists():
    raise FileNotFoundError("`gltf-transform` binary missing; run the install cell above before compressing.")

CSV_PATH = ROOT / "scripts" / "results-draco.csv"
SCENE_ROOT = ROOT / "data" / "scene" / "scene"

print("Repository root:", ROOT)
print("Scene directory:", SCENE_ROOT)
print("CSV:", CSV_PATH)


def ensure_csv_header(csv_path: Path) -> None:
    if not csv_path.exists():
        csv_path.parent.mkdir(parents=True, exist_ok=True)
        csv_path.write_text("path,before_bytes,after_bytes,saved_bytes,saved_pct,action,qpos,qnorm,qtex,timestamp\n")


def run_quantize(glb_path: Path, qfile: Path, qpos: int, qnorm: int, qtex: int) -> None:
    subprocess.run(
        [
            str(BIN),
            "quantize",
            str(glb_path),
            str(qfile),
            "--quantize-position",
            str(qpos),
            "--quantize-normal",
            str(qnorm),
            "--quantize-texcoord",
            str(qtex),
        ],
        check=True,
    )


def run_draco(qfile: Path, tmp: Path) -> None:
    subprocess.run([str(BIN), "draco", str(qfile), str(tmp)], check=True)


def write_csv_row(csv_path: Path, row: list) -> None:
    ensure_csv_header(csv_path)
    with csv_path.open("a", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(row)


def compress_glb(
    glb_path: Path,
    qpos: int = 14,
    qnorm: int = 10,
    qtex: int = 12,
    dry: bool = False,
) -> None:
    print(f"Compressing: {glb_path}")
    before = glb_path.stat().st_size
    qfile = glb_path.with_name(f"{glb_path.stem}.q.glb")
    tmp = glb_path.with_name(f"{glb_path.stem}.tmp.glb")

    try:
        run_quantize(glb_path, qfile, qpos, qnorm, qtex)
        run_draco(qfile, tmp)
        after = tmp.stat().st_size if tmp.exists() else 0

        saved = 0
        pct = 0
        action = "no_gain"

        if 0 < after < before:
            saved = before - after
            pct = (100 * saved) // before
            if dry:
                action = "dry_replace"
                tmp.unlink(missing_ok=True)
                print(f"DRY_RUN: would shrink {before} → {after} bytes ({pct}%)")
            else:
                tmp.replace(glb_path)
                action = "replaced"
                print(f"Replaced {before} → {after} bytes ({pct}%)")
        else:
            if tmp.exists():
                tmp.unlink(missing_ok=True)
            print("No gain: keeping original file")

        write_csv_row(
            CSV_PATH,
            [
                str(glb_path),
                before,
                after,
                saved,
                pct,
                action,
                qpos,
                qnorm,
                qtex,
                datetime.now().astimezone().strftime("%Y-%m-%dT%H:%M:%S%z"),
            ],
        )
    finally:
        qfile.unlink(missing_ok=True)


def compress_scene_dir(
    root: Path,
    qpos: int = 14,
    qnorm: int = 10,
    qtex: int = 12,
    dry: bool = False,
    max_files: Optional[int] = None,
) -> None:
    if not root.exists():
        raise FileNotFoundError(f"Scene directory {root} does not exist")

    ensure_csv_header(CSV_PATH)
    files = sorted(root.rglob("*.glb"))
    processed = 0

    for glb_path in files:
        if glb_path.name.endswith((".q.glb", ".tmp.glb")):
            continue
        compress_glb(glb_path, qpos=qpos, qnorm=qnorm, qtex=qtex, dry=dry)
        processed += 1
        if max_files is not None and processed >= max_files:
            break

    print(f"\nProcessed {processed} GLB file(s). CSV at {CSV_PATH}")



FileNotFoundError: Could not locate package.json; run this notebook from inside the repo.

In [ ]:
# Flip this to False once you are happy with the dry-run stats.
DRY_RUN = True
compress_scene_dir(SCENE_ROOT, dry=DRY_RUN)

# Example: target one topic or limit the number of files
# compress_scene_dir(SCENE_ROOT / "animal", dry=True, max_files=5)

